In [1]:
!pip -q install gensim

In [2]:
import re
import string
from gensim.models import Word2Vec
from gensim.corpora.wikicorpus import WikiCorpus
import os
from typing import List
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score

# Part 1: Train Arabic Word Embeddings (Word2Vec)



1. Collect Dataset





In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# the dataset cosists of Arabic Wikipedia dumps
wiki_path = "/content/drive/MyDrive/SDAIA Arabic NLP Bootcamp/Project/arwiki-latest-pages-articles.xml.bz2"

if not os.path.exists(wiki_path):
    raise FileNotFoundError("file not found")

wiki = WikiCorpus(wiki_path, dictionary={})

2. Preprocessing (Keeping only Arabic tokens)



In [5]:
def preprocess_arabic_token(tokens: List):

    tashkeel_pattern = re.compile(r'[\u0617-\u061A\u064B-\u0652]') #remove tashkeel

    arabic_punct = "،؛؟«»ـ…"
    punct_pattern = f"[{re.escape(string.punctuation + arabic_punct)}]"

    non_arabic_pattern = re.compile(r"[^\u0600-\u06FF]+") #remove non-Arabic tokens

    cleaned_tokens = []

    for token in tokens:

        print(f"token before: {token}")
        token = re.sub(r"[أإآ]", "ا", token)
        token = re.sub(r"ى", "ي", token)
        token = re.sub(r"ة", "ه", token)

        token = re.sub(tashkeel_pattern, "", token)

        token = re.sub(punct_pattern, "", token)

        token = re.sub(non_arabic_pattern, "", token)

        print(f"token after: {token}")

        if token:
            cleaned_tokens.append(token)

    return cleaned_tokens

In [6]:
# Check before preprocessing
tokenized_corpus = []
samples = 500_000

for i, tokens in enumerate(wiki.get_texts()):
    # print(len(tokens), tokens)
    tokenized_corpus.append(tokens)
    if i + 1 >= samples:
        break


Process InputQueue-2:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/lib/python3.12/dist-packages/gensim/utils.py", line 1291, in run
    wrapped_chunk = [list(chunk)]
                     ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gensim/corpora/wikicorpus.py", line 685, in <genexpr>
    in extract_pages(bz2.BZ2File(self.fname), self.filter_namespaces, self.filter_articles)
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gensim/corpora/wikicorpus.py", line 421, in extract_pages
    for elem in elems:
                ^^^^^


In [ ]:
cleaned_corpus = []

for token in tokenized_corpus:
  cleaned_tokens = preprocess_arabic_token(token)
  #print(len(cleaned_tokens), cleaned_tokens)
  cleaned_corpus.append(cleaned_tokens)



Streaming output truncated to the last 5000 lines.
token after: دور
token before: السينما
token after: السينما
token before: مطلع
token after: مطلع
token before: تسعينات
token after: تسعينات
token before: القرن
token after: القرن
token before: العشرين
token after: العشرين
token before: نتيجة
token after: نتيجه
token before: صعود
token after: صعود
token before: الأحزاب
token after: الاحزاب
token before: السياسية
token after: السياسيه
token before: الدينية
token after: الدينيه
token before: يعد
token after: يعد
token before: مسلسل
token after: مسلسل
token before: حكايات
token after: حكايات
token before: دحباش
token after: دحباش
token before: من
token after: من
token before: أول
token after: اول
token before: المسلسلات
token after: المسلسلات
token before: اليمنية
token after: اليمنيه
token before: وأدى
token after: وادي
token before: دور
token after: دور
token before: الشخصية
token after: الشخصيه
token before: دحباش
token after: دحباش
token before: الممثل
token after: الممثل
token before:

In [ ]:
# Train Word2Vec with the required settings
w2v_model = Word2Vec(
    sentences=cleaned_corpus,
    sg=1,             # Skip-gram
    vector_size=100,  # Embedding size
    window=5,         # Context window
    min_count=2,      # Keep words with freq >= 2
    workers=1,
    epochs=10
)

# Vocabulary size
print("Vocabulary size:", len(w2v_model.wv.index_to_key))

In [ ]:
# Show similar words for 3 Arabic words
query_words = ["لغة", "العربية", "تعلم"]

for word in query_words:
    if word in w2v_model.wv:
        print(f"Most similar words to '{word}':")
        print(w2v_model.wv.most_similar(word, topn=5))
    else:
        print(f"'{word}' is not in vocabulary (because of min_count=2)")
    print("-" * 40)

## Compare Word2Vec Settings\n
\n
Now we compare simple Word2Vec setups:\n
- CBOW (`sg=0`) vs Skip-gram (`sg=1`)\n
- `vector_size=50` vs `vector_size=100`\n
\n
We use a tiny score: average similarity of nearest words for a few query words.

In [ ]:
def train_and_score(tokenized_sentences, sg, vector_size):
    # Train one Word2Vec model with given settings
    model = Word2Vec(
        sentences=tokenized_sentences,
        sg=sg,
        vector_size=vector_size,
        window=5,
        min_count=2,
        workers=1,
        epochs=100
    )

    # Simple score: average top-1 similarity for a few words
    check_words = ["تعلم", "تحليل", "النصوص"]
    sims = []

    for w in check_words:
        if w in model.wv:
            nearest = model.wv.most_similar(w, topn=1)
            sims.append(nearest[0][1])

    # If no words were found, score is 0
    score = float(np.mean(sims)) if len(sims) > 0 else 0.0
    return model, score

# 4 simple configs
configs = [
    ("CBOW-50", 0, 50),
    ("CBOW-100", 0, 100),
    ("SkipGram-50", 1, 50),
    ("SkipGram-100", 1, 100)
]

comparison_results = []

for name, sg, size in configs:
    _, score = train_and_score(tokenized_corpus, sg=sg, vector_size=size)
    comparison_results.append((name, score))
    print(f"{name} -> score: {score:.4f}")

# Pick best one by score
best_name, best_score = max(comparison_results, key=lambda x: x[1])
print("\nBest config:", best_name, f"(score={best_score:.4f})")

# Short human-like observations
print("\nObservations:")
print("- In this small dataset,", best_name, "performed a bit better.")
print("- Skip-gram usually helps rare words more, while CBOW is often more stable.")
print("- Bigger vector size (100) can capture more detail, but with tiny data the gain may be small.")

## Part 2: Sentence Representation

Now we convert each tokenized sentence (list of words) into one vector by averaging word vectors.

In [ ]:
def sentence_to_vector(words, model):
    """words: list of words -> one average vector"""

    # Take vectors for words found in vocabulary
    vectors = [model.wv[word] for word in words if word in model.wv]

    # If sentence has no known words, return zeros
    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    # Average word vectors
    return np.mean(vectors, axis=0)

# Apply to all sentences and store in a list
sentence_vectors_list = [sentence_to_vector(words, w2v_model) for words in cleaned_corpus]

# Convert to NumPy array (useful for training later)
sentence_vectors = np.array(sentence_vectors_list)

print("Number of sentence vectors:", len(sentence_vectors_list))
print("Sentence vectors shape:", sentence_vectors.shape)
print("First vector (first 5 values):")
print(sentence_vectors[0][:5])

## Part 3: Text Classification (PyTorch)

We will build a tiny classifier on top of sentence vectors.

Labels:
- 0 = negative
- 1 = positive

In [ ]:
# Example labels (just for learning)
# 0 = negative, 1 = positive
# Note: Arabic wiki corpus is unlabeled, so these are toy labels for demo only
demo_size = min(200, len(sentence_vectors))
sentence_vectors_demo = sentence_vectors[:demo_size]

# Simple fake labels just to practice classification code
# (In real projects, use real labeled data)
labels = np.array([1 if i % 2 == 0 else 0 for i in range(demo_size)])

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    sentence_vectors_demo,
    labels,
    test_size=0.25,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [ ]:
class SimpleTextDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

# DataLoader for training
train_dataset = SimpleTextDataset(X_train_t, y_train_t)
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)

In [ ]:
class SimpleClassifier(nn.Module):
    """Simple NN: input -> hidden -> output"""
    def __init__(self, input_dim, num_classes):
        super().__init__()

        # Layer 1: from 100-dim sentence vector to hidden layer
        self.fc1 = nn.Linear(input_dim, 64)

        # Activation
        self.relu = nn.ReLU()

        # Layer 2: hidden layer to 2 classes
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Build model
model = SimpleClassifier(input_dim=100, num_classes=2)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Simple training loop (not production, just learning)
num_epochs = 15

for epoch in range(num_epochs):
    # Training mode
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_dataloader:
        # 1) Forward
        outputs = model(batch_x)

        # 2) Loss
        loss = criterion(outputs, batch_y)

        # 3) Backward + update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Print average loss per epoch
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

In [ ]:
# Evaluate on test set
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    predictions = torch.argmax(logits, dim=1)
    accuracy = (predictions == y_test_t).float().mean().item()

# Convert tensors to Python lists for sklearn metrics
y_true = y_test_t.tolist()
y_pred = predictions.tolist()

# Optional metrics
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("=== Test Results ===")
print("Predictions:", predictions.tolist())
print("True labels:", y_test_t.tolist())
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1-score : {f1:.2f}")

## Final Notes

This is a beginner pipeline:
1. Preprocess Arabic text
2. Train Word2Vec
3. Build sentence vectors
4. Train a simple classifier

You can improve it later with more data and better labels.